In [1]:
import numpy as np
import pandas as pd
import pycountry
from scipy import stats
import statsmodels.api as sm
import pyfixest as pf
from statsmodels.iolib.summary2 import summary_col
from statsmodels.stats.multitest import multipletests
import gc
import itertools

# Parallel trends assumption

In [ ]:
output_csv = "../../results/figure_data/data_figure_s1.csv"

df = pd.read_csv("../../data/processed/paper_level_analysis.csv")
df["accepted_date"] = pd.to_datetime(df["accepted_date"], errors="coerce", format="mixed")
df["ym"] = df["accepted_date"].dt.to_period("M")
df["ym_fe"] = df["ym"].astype(str).astype("category")
df["subfield"] = df["paper_subfield"].astype("category")
df["journal_id"] = df["journal_id"].astype("category")

policy_start = pd.Period("2023-01", freq="M")
base_period = -1

df["event_time"] = (df["ym"] - policy_start).apply(
    lambda x: x.n if pd.notna(x) else pd.NA
)

df["first_author_english_country"] = pd.to_numeric(
    df["first_author_english_country"], errors="coerce"
)
df["last_author_english_country"] = pd.to_numeric(
    df["last_author_english_country"], errors="coerce"
)

event_times = sorted(df["event_time"].dropna().astype(int).unique())

first_cols = []
last_cols = []
first_author_interactions = []
last_author_interactions = []

for t in event_times:
    if t == base_period:
        continue

    suffix = f"m{abs(t)}" if t < 0 else str(t)
    event_indicator = (df["event_time"] == t).astype(int)

    first_col = f"first_x_event_{suffix}"
    last_col = f"last_x_event_{suffix}"

    first_cols.append(first_col)
    last_cols.append(last_col)

    first_author_interactions.append(
        (df["first_author_english_country"] * event_indicator).rename(first_col)
    )
    last_author_interactions.append(
        (df["last_author_english_country"] * event_indicator).rename(last_col)
    )

if first_author_interactions:
    df = pd.concat([df, pd.concat(first_author_interactions, axis=1)], axis=1)

if last_author_interactions:
    df = pd.concat([df, pd.concat(last_author_interactions, axis=1)], axis=1)

controls = [
    "paper_authors_count",
    "paper_ref_count",
    "sentences_intro_discussion",
]
control_formula = " + ".join(controls)

formula_first = (
    "ai_use_intro_discussion ~ "
    "first_author_english_country + "
    + " + ".join(first_cols)
    + " + "
    + control_formula
    + " | journal_id + ym_fe + subfield"
)

formula_last = (
    "ai_use_intro_discussion ~ "
    "last_author_english_country + "
    + " + ".join(last_cols)
    + " + "
    + control_formula
    + " | journal_id + ym_fe + subfield"
)

ev_first = pf.feols(
    fml=formula_first,
    data=df,
    vcov={"CRV1": "journal_id"},
)

ev_last = pf.feols(
    fml=formula_last,
    data=df,
    vcov={"CRV1": "journal_id"},
)


def extract_event_results(model, terms, prefix, policy_start):
    coef = model.coef()
    ci = model.confint()
    rows = []

    for term in terms:
        if term not in coef.index:
            continue

        merge_key = term.replace(prefix, "")
        event_time_num = (
            -int(merge_key[1:]) if merge_key.startswith("m") else int(merge_key)
        )
        year_month = str(policy_start + event_time_num)

        rows.append(
            {
                "merge_key": merge_key,
                "event_time": event_time_num,
                "year_month": year_month,
                "coef": coef.loc[term],
                "coef_low": ci.loc[term].iloc[0],
                "coef_high": ci.loc[term].iloc[1],
            }
        )

    return pd.DataFrame(rows)


first_results = extract_event_results(
    ev_first,
    first_cols,
    "first_x_event_",
    policy_start,
).rename(
    columns={
        "coef": "first_coef",
        "coef_low": "first_coef_low",
        "coef_high": "first_coef_high",
    }
)

last_results = extract_event_results(
    ev_last,
    last_cols,
    "last_x_event_",
    policy_start,
).rename(
    columns={
        "coef": "last_coef",
        "coef_low": "last_coef_low",
        "coef_high": "last_coef_high",
    }
)

results = pd.merge(
    first_results,
    last_results,
    on=["merge_key", "event_time", "year_month"],
    how="outer",
)

results = results.sort_values("event_time")[
    [
        "year_month",
        "first_coef",
        "first_coef_low",
        "first_coef_high",
        "last_coef",
        "last_coef_low",
        "last_coef_high",
    ]
]

results.to_csv(output_csv, index=False)

C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1011 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 964 singleton fixed effect(s) dropped from the model.
  warnings.warn(


# Difference-in-Differences analysis

In [ ]:
INPUT_FILE = "../../data/processed/paper_level_analysis.csv"
RESULT_TABLE_FILE = "../../results/tables/regression_results_did.csv"

def significance_stars(pvalue):
    if pd.isna(pvalue):
        return ""
    if pvalue < 0.001:
        return "***"
    if pvalue < 0.01:
        return "**"
    if pvalue < 0.05:
        return "*"
    return ""

def get_series_from_result(result, method_name, tidy_col):
    if hasattr(result, method_name):
        method = getattr(result, method_name)
        try:
            out = method()
            if isinstance(out, pd.Series):
                return out
        except Exception:
            pass
    tidy_df = result.tidy().copy()
    if "Coefficient" in tidy_df.columns:
        tidy_df = tidy_df.set_index("Coefficient")
    if tidy_col in tidy_df.columns:
        return tidy_df[tidy_col]
    raise ValueError(f"Cannot extract column '{tidy_col}' from pyfixest result.")

def get_nobs(result):
    for attr in ["_N", "N", "nobs"]:
        if hasattr(result, attr):
            value = getattr(result, attr)
            if callable(value):
                try:
                    value = value()
                except Exception:
                    continue
            try:
                return int(value)
            except Exception:
                continue
    return np.nan

def get_r2(result):
    for attr in ["_r2", "_r2_overall", "r2", "rsquared"]:
        if hasattr(result, attr):
            value = getattr(result, attr)
            if callable(value):
                try:
                    value = value()
                except Exception:
                    continue
            try:
                return float(value)
            except Exception:
                continue
    return np.nan

def validate_binary_variable(data, column):
    observed_values = set(pd.to_numeric(data[column], errors="coerce").dropna().unique())
    invalid_values = observed_values.difference({0, 1})
    if invalid_values:
        raise ValueError(
            f"{column} must be coded as 0/1. "
            f"Observed values: {sorted(observed_values)}"
        )

def prepare_category(data, column):
    data[column] = (
        data[column]
        .astype("string")
        .fillna(f"__missing_{column}__")
        .astype("category")
    )

def calculate_raw_did(data, treatment_col, outcome_col):
    means = data.groupby(
        [treatment_col, "post_gpt"],
        observed=True,
    )[outcome_col].mean()
    required_cells = [(0, 0), (0, 1), (1, 0), (1, 1)]
    missing_cells = [cell for cell in required_cells if cell not in means.index]
    if missing_cells:
        return np.nan
    control_pre = means.loc[(0, 0)]
    control_post = means.loc[(0, 1)]
    treated_pre = means.loc[(1, 0)]
    treated_post = means.loc[(1, 1)]
    raw_did = treated_post - treated_pre - control_post + control_pre
    return raw_did

df = pd.read_csv(INPUT_FILE, low_memory=False)

required_input_columns = [
    "accepted_date",
    "ai_use_intro_discussion",
    "journal_id",
    "paper_subfield",
    "first_author_english_country",
    "last_author_english_country",
    "first_author_id",
    "last_author_id",
    "paper_authors_count",
    "paper_ref_count",
    "sentences_intro_discussion",
]

missing_input_columns = [
    col for col in required_input_columns
    if col not in df.columns
]

if missing_input_columns:
    raise ValueError(
        "The following required columns are missing: "
        + ", ".join(missing_input_columns)
    )

df["accepted_date"] = pd.to_datetime(df["accepted_date"], errors="coerce", format='mixed')
df = df.dropna(subset=["accepted_date"]).copy()

cutoff_date = pd.Timestamp("2023-01-01")

df["post_gpt"] = (df["accepted_date"] >= cutoff_date).astype("int8")

numeric_cols = [
    "ai_use_intro_discussion",
    "first_author_english_country",
    "last_author_english_country",
    "paper_authors_count",
    "paper_ref_count",
    "sentences_intro_discussion",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

validate_binary_variable(df, "first_author_english_country")
validate_binary_variable(df, "last_author_english_country")

prepare_category(df, "journal_id")
prepare_category(df, "paper_subfield")

df["first_author_id"] = df["first_author_id"].astype("string")
df["last_author_id"] = df["last_author_id"].astype("string")

df["did_first"] = df["post_gpt"] * df["first_author_english_country"]
df["did_last"] = df["post_gpt"] * df["last_author_english_country"]

df = df.replace([np.inf, -np.inf], np.nan)

df["log_paper_authors_count"] = np.log(df["paper_authors_count"] + 1)
df["log_paper_ref_count"] = np.log(df["paper_ref_count"] + 1)
df["log_sentences_intro_discussion"] = np.log(df["sentences_intro_discussion"] + 1)

base_controls = [
    "log_paper_authors_count",
    "log_paper_ref_count",
    "log_sentences_intro_discussion",
]

common_required_cols = [
    "ai_use_intro_discussion",
    "post_gpt",
    "journal_id",
    "paper_subfield",
] + base_controls

first_required_cols = common_required_cols + [
    "first_author_english_country",
    "did_first",
]

last_required_cols = common_required_cols + [
    "last_author_english_country",
    "did_last",
]

df_first = df.dropna(subset=first_required_cols).copy()
df_last = df.dropna(subset=last_required_cols).copy()

first_float_cols = [
    "ai_use_intro_discussion",
    "first_author_english_country",
    "did_first",
    "log_paper_authors_count",
    "log_paper_ref_count",
    "log_sentences_intro_discussion",
]

last_float_cols = [
    "ai_use_intro_discussion",
    "last_author_english_country",
    "did_last",
    "log_paper_authors_count",
    "log_paper_ref_count",
    "log_sentences_intro_discussion",
]

for col in first_float_cols:
    df_first[col] = df_first[col].astype("float64")

for col in last_float_cols:
    df_last[col] = df_last[col].astype("float64")

df_first["post_gpt"] = df_first["post_gpt"].astype("int8")
df_last["post_gpt"] = df_last["post_gpt"].astype("int8")

df_first_author_fe = df_first.dropna(subset=["first_author_id"]).copy()
df_last_author_fe = df_last.dropna(subset=["last_author_id"]).copy()

df_first_author_fe["first_author_id"] = df_first_author_fe["first_author_id"].astype("category")
df_last_author_fe["last_author_id"] = df_last_author_fe["last_author_id"].astype("category")

first_group_means = (
    df_first.groupby(
        ["first_author_english_country", "post_gpt"],
        observed=True,
    )["ai_use_intro_discussion"]
    .agg(["mean", "count", "std"])
    .reset_index()
)

first_group_means["model"] = "First author"

last_group_means = (
    df_last.groupby(
        ["last_author_english_country", "post_gpt"],
        observed=True,
    )["ai_use_intro_discussion"]
    .agg(["mean", "count", "std"])
    .reset_index()
)

last_group_means["model"] = "Last author"

raw_did_first = calculate_raw_did(
    data=df_first,
    treatment_col="first_author_english_country",
    outcome_col="ai_use_intro_discussion",
)

raw_did_last = calculate_raw_did(
    data=df_last,
    treatment_col="last_author_english_country",
    outcome_col="ai_use_intro_discussion",
)

formula_first_nofe = (
    "ai_use_intro_discussion ~ "
    "post_gpt + "
    "first_author_english_country + "
    "did_first + "
    "log_paper_authors_count + "
    "log_paper_ref_count + "
    "log_sentences_intro_discussion"
)

formula_last_nofe = (
    "ai_use_intro_discussion ~ "
    "post_gpt + "
    "last_author_english_country + "
    "did_last + "
    "log_paper_authors_count + "
    "log_paper_ref_count + "
    "log_sentences_intro_discussion"
)

formula_first_jsfe = (
    "ai_use_intro_discussion ~ "
    "post_gpt + "
    "first_author_english_country + "
    "did_first + "
    "log_paper_authors_count + "
    "log_paper_ref_count + "
    "log_sentences_intro_discussion "
    "| journal_id + paper_subfield"
)

formula_last_jsfe = (
    "ai_use_intro_discussion ~ "
    "post_gpt + "
    "last_author_english_country + "
    "did_last + "
    "log_paper_authors_count + "
    "log_paper_ref_count + "
    "log_sentences_intro_discussion "
    "| journal_id + paper_subfield"
)

formula_first_authorjsfe = (
    "ai_use_intro_discussion ~ "
    "post_gpt + "
    "did_first + "
    "log_paper_authors_count + "
    "log_paper_ref_count + "
    "log_sentences_intro_discussion "
    "| journal_id + paper_subfield + first_author_id"
)

formula_last_authorjsfe = (
    "ai_use_intro_discussion ~ "
    "post_gpt + "
    "did_last + "
    "log_paper_authors_count + "
    "log_paper_ref_count + "
    "log_sentences_intro_discussion "
    "| journal_id + paper_subfield + last_author_id"
)


model_specs = [
    ("First author (No FE)", formula_first_nofe, df_first),
    ("First author (JS FE)", formula_first_jsfe, df_first),
    ("First author (JS+Author FE)", formula_first_authorjsfe, df_first_author_fe),
    ("Last author (No FE)", formula_last_nofe, df_last),
    ("Last author (JS FE)", formula_last_jsfe, df_last),
    ("Last author (JS+Author FE)", formula_last_authorjsfe, df_last_author_fe),
]

keep_vars = [
    "post_gpt",
    "did_first",
    "did_last",
    "first_author_english_country",
    "last_author_english_country",
    "log_paper_authors_count",
    "log_paper_ref_count",
    "log_sentences_intro_discussion",
]

table_rows = []

for var in keep_vars:
    table_rows.append(var)
    table_rows.append(f"({var})")

final_table = pd.DataFrame(
    index=table_rows,
    columns=[model_name for model_name, _, _ in model_specs],
    dtype=object,
)

extra_row_names = [
    "Base controls",
    "Post main effect",
    "Journal FE",
    "Subfield FE",
    "Author FE",
    "Clustered SE",
    "N",
    "R2",
]

extra_rows = pd.DataFrame(
    index=extra_row_names,
    columns=final_table.columns,
    dtype=object,
)

for model_name, formula, model_data in model_specs:
    result = pf.feols(
        fml=formula,
        data=model_data,
        vcov={"CRV1": "journal_id"},
    )

    params = get_series_from_result(result, "coef", "Estimate")
    std_errors = get_series_from_result(result, "se", "Std. Error")
    pvalues = get_series_from_result(result, "pvalue", "Pr(>|t|)")

    for var in keep_vars:
        coefficient_row = var
        standard_error_row = f"({var})"
        if var in params.index:
            coefficient = params.loc[var]
            standard_error = std_errors.loc[var]
            pvalue = pvalues.loc[var]
            final_table.loc[coefficient_row, model_name] = (
                f"{coefficient:.4f}"
                f"{significance_stars(pvalue)}"
            )
            final_table.loc[standard_error_row, model_name] = f"({standard_error:.4f})"
        else:
            final_table.loc[coefficient_row, model_name] = ""
            final_table.loc[standard_error_row, model_name] = ""

    extra_rows.loc["Base controls", model_name] = "Yes"
    extra_rows.loc["Post main effect", model_name] = "Yes"
    extra_rows.loc["Clustered SE", model_name] = "Journal"

    if "JS+Author FE" in model_name:
        extra_rows.loc["Journal FE", model_name] = "Yes"
        extra_rows.loc["Subfield FE", model_name] = "Yes"
        extra_rows.loc["Author FE", model_name] = "Yes"
    elif "JS FE" in model_name:
        extra_rows.loc["Journal FE", model_name] = "Yes"
        extra_rows.loc["Subfield FE", model_name] = "Yes"
        extra_rows.loc["Author FE", model_name] = "No"
    elif "No FE" in model_name:
        extra_rows.loc["Journal FE", model_name] = "No"
        extra_rows.loc["Subfield FE", model_name] = "No"
        extra_rows.loc["Author FE", model_name] = "No"

    nobs = get_nobs(result)
    if pd.isna(nobs):
        extra_rows.loc["N", model_name] = ""
    else:
        extra_rows.loc["N", model_name] = f"{int(nobs):,}"

    r2_value = get_r2(result)
    if pd.isna(r2_value):
        extra_rows.loc["R2", model_name] = ""
    else:
        extra_rows.loc["R2", model_name] = f"{r2_value:.4f}"

    del params, std_errors, pvalues, result
    gc.collect()

final_table = pd.concat([
    final_table,
    extra_rows,
])

C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1011 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1014622 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 964 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 559748 singleton fixed effect(s) dropped from the model.
  warnings.warn(


In [10]:
final_table.to_csv(
    RESULT_TABLE_FILE
)

# Difference-in-difference-in-differences analysis

In [ ]:
INPUT_FILE = "../../data/processed/paper_level_analysis.csv"
POST_CUTOFF = "2023-01-01"

USE_NAME_FOR_ENGLISH_COUNTRY = False
FOCAL_SECTION = "intro_discussion" # or method_results

OUTCOME = "ai_use_" + FOCAL_SECTION
SENTENCE_COUNT_COL = "sentences_" + FOCAL_SECTION

country_tag = (
    "name_nationality"
    if USE_NAME_FOR_ENGLISH_COUNTRY
    else "affiliation_country"
)
section_tag = FOCAL_SECTION

out_file_suffix = f"{country_tag}_journal_subfield_fe_{section_tag}"

GROUP_AI_RELATIVE_CHANGE_CSV = (
    f"../../results/tables/group_ai_relative_change_{out_file_suffix}.csv"
)
DDD_ALL_COEFS_CSV = (
    f"../../results/tables/ddd_all_coefficients_{out_file_suffix}.csv"
)
DDD_GROUP_PAIRWISE_CSV = (
    f"../../results/tables/ddd_group_pairwise_comparisons_{out_file_suffix}.csv"
)

Z_975 = 1.959963984540054

COUNTRY_ENGLISH_COLS = {
    "first": (
        "first_author_english_country_author_name"
        if USE_NAME_FOR_ENGLISH_COUNTRY
        else "first_author_english_country"
    ),
    "last": (
        "last_author_english_country_author_name"
        if USE_NAME_FOR_ENGLISH_COUNTRY
        else "last_author_english_country"
    ),
}

def make_log_control(df, raw_col, log_col):
    df[log_col] = np.log1p(pd.to_numeric(df[raw_col], errors="coerce"))
    return df

CONTROL_RAW = [
    "paper_authors_count",
    "paper_ref_count",
    SENTENCE_COUNT_COL,
]

CONTROL_VARS = [
    "log_paper_authors_count",
    "log_paper_ref_count",
    "log_sentences_count",
]

df = pd.read_csv(INPUT_FILE, low_memory=False)

required_cols = [
    "paper_id",
    "accepted_date",
    OUTCOME,
    SENTENCE_COUNT_COL,
    "paper_ref_count",
    "paper_authors_count",
    "paper_subfield",
    "journal_id",
    COUNTRY_ENGLISH_COLS["first"],
    COUNTRY_ENGLISH_COLS["last"],
    "first_author_age",
    "last_author_age",
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise KeyError(f"Missing required columns: {missing}")

df["accepted_date"] = pd.to_datetime(df["accepted_date"], errors="coerce")

df["post_gpt"] = np.where(
    df["accepted_date"].notna(),
    (df["accepted_date"] >= pd.to_datetime(POST_CUTOFF)).astype("int8"),
    np.nan,
)

df["paper_journal_fe"] = (
    df["journal_id"]
    .astype("string")
    .fillna("__missing_journal__")
    .astype("category")
)

df["paper_subfield_fe"] = (
    df["paper_subfield"]
    .astype("string")
    .fillna("__missing_subfield__")
    .astype("category")
)

df = make_log_control(df, "paper_authors_count", "log_paper_authors_count")
df = make_log_control(df, "paper_ref_count", "log_paper_ref_count")
df = make_log_control(df, SENTENCE_COUNT_COL, "log_sentences_count")

df[COUNTRY_ENGLISH_COLS["first"]] = pd.to_numeric(
    df[COUNTRY_ENGLISH_COLS["first"]],
    errors="coerce",
)
df[COUNTRY_ENGLISH_COLS["last"]] = pd.to_numeric(
    df[COUNTRY_ENGLISH_COLS["last"]],
    errors="coerce",
)

df["english_first"] = df[COUNTRY_ENGLISH_COLS["first"]]
df["english_last"] = df[COUNTRY_ENGLISH_COLS["last"]]

df["first_author_age"] = pd.to_numeric(
    df["first_author_age"],
    errors="coerce",
)

df["last_author_age"] = pd.to_numeric(
    df["last_author_age"],
    errors="coerce",
)

for level in [10, 15, 20]:
    df[f"first_senior_{level}"] = np.where(
        df["first_author_age"].notna(),
        (df["first_author_age"] >= level).astype("int8"),
        np.nan,
    )
    df[f"last_senior_{level}"] = np.where(
        df["last_author_age"].notna(),
        (df["last_author_age"] >= level).astype("int8"),
        np.nan,
    )

specs = []
raw_top_vars = []

for level in [1, 5, 10, 20]:
    first_top = f"first_cumulative_pubs_top{level}"
    last_top = f"last_cumulative_pubs_top{level}"
    raw_top_vars.extend([first_top, last_top])
    specs.append({
        "indicator": "pub",
        "order": "first",
        "level": level,
        "eng_var": "english_first",
        "top_var": first_top,
    })
    specs.append({
        "indicator": "pub",
        "order": "last",
        "level": level,
        "eng_var": "english_last",
        "top_var": last_top,
    })

for level in [1, 5, 10, 20]:
    first_top = f"first_cumulative_cits_top{level}"
    last_top = f"last_cumulative_cits_top{level}"
    raw_top_vars.extend([first_top, last_top])
    specs.append({
        "indicator": "cit",
        "order": "first",
        "level": level,
        "eng_var": "english_first",
        "top_var": first_top,
    })
    specs.append({
        "indicator": "cit",
        "order": "last",
        "level": level,
        "eng_var": "english_last",
        "top_var": last_top,
    })

for level in [10, 15, 20]:
    first_top = f"first_senior_{level}"
    last_top = f"last_senior_{level}"
    raw_top_vars.extend([first_top, last_top])
    specs.append({
        "indicator": "age",
        "order": "first",
        "level": level,
        "eng_var": "english_first",
        "top_var": first_top,
    })
    specs.append({
        "indicator": "age",
        "order": "last",
        "level": level,
        "eng_var": "english_last",
        "top_var": last_top,
    })

for level in [100, 200, 300]:
    first_top = f"first_top{level}_ins"
    last_top = f"last_top{level}_ins"
    raw_top_vars.extend([first_top, last_top])
    specs.append({
        "indicator": "ins",
        "order": "first",
        "level": level,
        "eng_var": "english_first",
        "top_var": first_top,
    })
    specs.append({
        "indicator": "ins",
        "order": "last",
        "level": level,
        "eng_var": "english_last",
        "top_var": last_top,
    })

for level in [20, 10, 5]:
    first_top = f"first_recent2year_top{level}"
    last_top = f"last_recent2year_top{level}"
    raw_top_vars.extend([first_top, last_top])
    specs.append({
        "indicator": "pub_recent2year",
        "order": "first",
        "level": level,
        "eng_var": "english_first",
        "top_var": first_top,
    })
    specs.append({
        "indicator": "pub_recent2year",
        "order": "last",
        "level": level,
        "eng_var": "english_last",
        "top_var": last_top,
    })

raw_top_vars = sorted(set(raw_top_vars))

missing_top_vars = [
    col for col in raw_top_vars
    if col not in df.columns
]

if missing_top_vars:
    raise KeyError(
        "Missing top-group indicator columns: "
        f"{missing_top_vars}"
    )

for col in raw_top_vars:
    df[col] = pd.to_numeric(df[col], errors="coerce")

binary_cols = [
    "english_first",
    "english_last",
] + raw_top_vars

for col in binary_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

common_required = [
    OUTCOME,
    "post_gpt",
    "paper_journal_fe",
    "paper_subfield_fe",
] + CONTROL_VARS

df = df.replace([np.inf, -np.inf], np.nan)

for col in common_required + binary_cols:
    if (
        col in df.columns
        and col not in [
            "paper_journal_fe",
            "paper_subfield_fe",
        ]
    ):
        df[col] = pd.to_numeric(df[col], errors="coerce")

def get_nobs(result):
    if hasattr(result, "_N"):
        return int(result._N)
    perf = result.get_performance()
    for key in ["nobs", "N", "Observations"]:
        if key in perf:
            return int(perf[key])
    return np.nan

def get_r2(result):
    if hasattr(result, "_r2"):
        return float(result._r2)
    perf = result.get_performance()
    for key in ["r2", "R2", "R Squared"]:
        if key in perf:
            return float(perf[key])
    return np.nan

def _coef_index(result):
    return [str(name) for name in result.coef().index]

def _find_interaction_term(result, variables):
    candidates = {
        ":".join(order)
        for order in itertools.permutations(variables)
    }
    for term in _coef_index(result):
        if term in candidates:
            return term
    return None

def group_post_terms(english_value, top_value, eng_var, top_var):
    terms = {"post_gpt": 1.0}
    if english_value == 1:
        terms[":".join([eng_var, "post_gpt"])] = 1.0
    if top_value == 1:
        terms[":".join(["post_gpt", top_var])] = 1.0
    if english_value == 1 and top_value == 1:
        terms[":".join([eng_var, "post_gpt", top_var])] = 1.0
    return terms

def subtract_term_weights(terms_a, terms_b):
    keys = set(terms_a) | set(terms_b)
    differences = {
        key: float(terms_a.get(key, 0.0) - terms_b.get(key, 0.0))
        for key in keys
    }
    return {
        key: weight
        for key, weight in differences.items()
        if weight != 0.0
    }

def holm_adjust_valid_pvalues(values):
    numeric = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    adjusted = np.full(len(numeric), np.nan, dtype=float)
    valid = np.isfinite(numeric)
    if valid.any():
        adjusted[valid] = multipletests(
            numeric[valid],
            method="holm",
        )[1]
    return pd.Series(adjusted, index=values.index)

def lincom(result, raw_terms):
    coef = result.coef()
    coef_names = [str(name) for name in coef.index]
    coef_values = coef.to_numpy(dtype=float)
    coef_lookup = dict(zip(coef_names, coef_values))
    weights = np.zeros(len(coef_names), dtype=float)
    used_terms = []
    missing_terms = []

    for raw_term, weight in raw_terms.items():
        pieces = raw_term.split(":")
        if len(pieces) == 1:
            term = raw_term if raw_term in coef_lookup else None
        else:
            term = _find_interaction_term(result, pieces)
        if term is None:
            missing_terms.append(raw_term)
            continue
        position = coef_names.index(term)
        weights[position] += float(weight)
        used_terms.append(term)

    estimate = float(np.dot(weights, coef_values))
    vcov = np.asarray(result._vcov, dtype=float)
    variance = float(weights @ vcov @ weights)

    if np.isfinite(variance):
        std_error = np.sqrt(max(variance, 0.0))
    else:
        std_error = np.nan

    if np.isfinite(std_error):
        lower = estimate - Z_975 * std_error
        upper = estimate + Z_975 * std_error
    else:
        lower = np.nan
        upper = np.nan

    return {
        "estimate": estimate,
        "std_error": std_error,
        "lower": lower,
        "upper": upper,
        "used_terms": "; ".join(used_terms),
        "missing_or_absorbed_terms": "; ".join(missing_terms),
    }

def extract_all_non_fe_coefficients(
    result,
    indicator,
    order,
    level,
    outcome,
    eng_var,
    top_var,
    model_fe_type,
):
    tidy = (
        result
        .tidy()
        .reset_index()
        .rename(columns={"Coefficient": "term"})
    )
    rows = []

    for _, row in tidy.iterrows():
        term = str(row["term"])
        rows.append({
            "indicator": indicator,
            "order": order,
            "level": level,
            "outcome": outcome,
            "eng_var": eng_var,
            "top_var": top_var,
            "term": term,
            "estimate": row.get("Estimate", np.nan),
            "std_error": row.get("Std. Error", np.nan),
            "t_value": row.get("t value", np.nan),
            "pvalue": row.get("Pr(>|t|)", np.nan),
            "lower": row.get("2.5%", np.nan),
            "upper": row.get("97.5%", np.nan),
            "status": "success",
            "model_fe_type": model_fe_type,
        })

    return rows

group_ai_relative_change_rows = []
ddd_all_coef_rows = []
ddd_group_pairwise_rows = []

group_labels = {
    (0, 0): "Non-English, less-established",
    (1, 0): "English, less-established",
    (0, 1): "Non-English, more-established",
    (1, 1): "English, more-established",
}

FE_TERMS = [
    "paper_journal_fe",
    "paper_subfield_fe",
]

FE_FORMULA = " + ".join(FE_TERMS)
FE_TYPE_LABEL = "journal + subfield"

for idx, spec in enumerate(specs, start=1):
    indicator = spec["indicator"]
    order = spec["order"]
    level = spec["level"]
    eng_var = spec["eng_var"]
    top_var = spec["top_var"]

    needed_cols = [
        OUTCOME,
        "post_gpt",
        eng_var,
        top_var,
        "paper_journal_fe",
        "paper_subfield_fe",
    ] + CONTROL_VARS

    df_model = df[needed_cols].dropna().copy()

    if (
        df_model.empty
        or df_model[eng_var].nunique() < 2
        or df_model[top_var].nunique() < 2
    ):
        del df_model
        gc.collect()
        continue

    formula_ddd = (
        f"{OUTCOME} ~ "
        f"{eng_var} * post_gpt * {top_var} + "
        f"{' + '.join(CONTROL_VARS)} "
        f"| {FE_FORMULA}"
    )

    result_ddd = None

    try:
        result_ddd = pf.feols(
            fml=formula_ddd,
            data=df_model,
            vcov={"CRV1": "paper_journal_fe"},
        )

        nobs_ddd = get_nobs(result_ddd)
        r2_ddd = get_r2(result_ddd)

        group_terms = {}

        for english_value in [0, 1]:
            for top_value in [0, 1]:
                raw_terms = group_post_terms(
                    english_value=english_value,
                    top_value=top_value,
                    eng_var=eng_var,
                    top_var=top_var,
                )
                group_key = (english_value, top_value)
                group_terms[group_key] = raw_terms
                lincom_res = lincom(result_ddd, raw_terms)

                group_ai_relative_change_rows.append({
                    "indicator": indicator,
                    "order": order,
                    "level": level,
                    "english": english_value,
                    "top": top_value,
                    "group": group_labels[group_key],
                    "ai_post_relative_change": lincom_res["estimate"],
                    "std_error": lincom_res["std_error"],
                    "lower": lincom_res["lower"],
                    "upper": lincom_res["upper"],
                    "outcome": OUTCOME,
                    "eng_var": eng_var,
                    "top_var": top_var,
                    "model_fe_type": FE_TYPE_LABEL,
                    "n": nobs_ddd,
                    "r2": r2_ddd,
                })

        for (
            group_a,
            terms_a,
        ), (
            group_b,
            terms_b,
        ) in itertools.combinations(group_terms.items(), 2):
            contrast_terms = subtract_term_weights(terms_a, terms_b)
            contrast_res = lincom(result_ddd, contrast_terms)
            contrast_se = contrast_res["std_error"]

            if np.isfinite(contrast_se) and contrast_se > 0:
                z_value = contrast_res["estimate"] / contrast_se
                pvalue = 2.0 * stats.norm.sf(abs(z_value))
            else:
                z_value = np.nan
                pvalue = np.nan

            ddd_group_pairwise_rows.append({
                "indicator": indicator,
                "order": order,
                "level": level,
                "english_a": group_a[0],
                "top_a": group_a[1],
                "group_a": group_labels[group_a],
                "english_b": group_b[0],
                "top_b": group_b[1],
                "group_b": group_labels[group_b],
                "difference_in_post_change": contrast_res["estimate"],
                "std_error": contrast_se,
                "z_value": z_value,
                "pvalue": pvalue,
                "lower": contrast_res["lower"],
                "upper": contrast_res["upper"],
                "used_terms": contrast_res["used_terms"],
                "missing_or_absorbed_terms": contrast_res[
                    "missing_or_absorbed_terms"
                ],
                "outcome": OUTCOME,
                "eng_var": eng_var,
                "top_var": top_var,
                "model_fe_type": FE_TYPE_LABEL,
                "n": nobs_ddd,
                "r2": r2_ddd,
            })

        coef_rows = extract_all_non_fe_coefficients(
            result=result_ddd,
            indicator=indicator,
            order=order,
            level=level,
            outcome=OUTCOME,
            eng_var=eng_var,
            top_var=top_var,
            model_fe_type=FE_TYPE_LABEL,
        )

        for row in coef_rows:
            row["n"] = nobs_ddd
            row["r2"] = r2_ddd

        ddd_all_coef_rows.extend(coef_rows)

    except Exception as exc:
        pass

    finally:
        if result_ddd is not None:
            del result_ddd
        del df_model
        gc.collect()

group_ai_relative_change = pd.DataFrame(group_ai_relative_change_rows)
ddd_all_coefs = pd.DataFrame(ddd_all_coef_rows)
ddd_group_pairwise_comparisons = pd.DataFrame(ddd_group_pairwise_rows)

if not group_ai_relative_change.empty:
    group_ai_relative_change = (
        group_ai_relative_change
        .sort_values([
            "indicator",
            "order",
            "level",
            "english",
            "top",
        ])
        .reset_index(drop=True)
    )

if not ddd_all_coefs.empty:
    ddd_all_coefs = (
        ddd_all_coefs
        .sort_values([
            "indicator",
            "order",
            "level",
            "term",
        ])
        .reset_index(drop=True)
    )

if not ddd_group_pairwise_comparisons.empty:
    ddd_group_pairwise_comparisons = (
        ddd_group_pairwise_comparisons
        .sort_values([
            "indicator",
            "order",
            "level",
            "english_a",
            "top_a",
            "english_b",
            "top_b",
        ])
        .reset_index(drop=True)
    )
    ddd_group_pairwise_comparisons["pvalue_holm"] = (
        ddd_group_pairwise_comparisons
        .groupby(
            [
                "indicator",
                "order",
                "level",
            ],
            group_keys=False,
        )["pvalue"]
        .transform(holm_adjust_valid_pvalues)
    )

group_ai_relative_change.to_csv(
    GROUP_AI_RELATIVE_CHANGE_CSV,
    index=False,
    encoding="utf-8-sig",
)

ddd_all_coefs.to_csv(
    DDD_ALL_COEFS_CSV,
    index=False,
    encoding="utf-8-sig",
)

ddd_group_pairwise_comparisons.to_csv(
    DDD_GROUP_PAIRWISE_CSV,
    index=False,
    encoding="utf-8-sig",
)

C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 934 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 906 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 934 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 906 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppData\Roaming\Python\Python313\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 934 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Users\Jialin\AppD

# Author-level OLS regression

In [5]:
first_author_level_data = pd.read_csv('../../data/processed/first_author_analysis.csv')
last_author_level_data = pd.read_csv('../../data/processed/last_author_analysis.csv')

## AI author analysis

In [6]:
def regression(data):
    df_work = data.copy()
    y = pd.to_numeric(df_work['ai_use_change'], errors='coerce')
    control_vars = ['author_ins_top_100', 'cumulative_pubs', 'cumulative_cits', 'author_age', 'author_english_country']
    X_controls = df_work[control_vars].copy()

    for col in X_controls.columns:
        X_controls[col] = pd.to_numeric(X_controls[col], errors='coerce')

    X_controls['cumulative_pubs'] = np.log1p(X_controls['cumulative_pubs'])
    X_controls['cumulative_cits'] = np.log1p(X_controls['cumulative_cits'])

    subfield_dummies = pd.get_dummies(df_work['author_field'], prefix='subfield', drop_first=True).astype(float)

    X0 = pd.concat([X_controls, subfield_dummies], axis=1)
    X0 = sm.add_constant(X0).astype('float64')
    mask0 = (~y.isna()) & (~X0.isna().any(axis=1))
    model0 = sm.OLS(y.loc[mask0].astype('float64'), X0.loc[mask0]).fit()

    X1 = X_controls.copy()
    X1['ai_author'] = pd.to_numeric(df_work['ai_author'], errors='coerce')
    X1 = pd.concat([X1, subfield_dummies], axis=1)
    X1 = sm.add_constant(X1).astype('float64')
    mask1 = (~y.isna()) & (~X1.isna().any(axis=1))
    model1 = sm.OLS(y.loc[mask1].astype('float64'), X1.loc[mask1]).fit()

    X2 = X_controls.copy()
    X2['ai_author'] = pd.to_numeric(df_work['ai_author'], errors='coerce')
    X2['ai_author_english'] = X2['ai_author'] * X2['author_english_country']
    X2 = pd.concat([X2, subfield_dummies], axis=1)
    X2 = sm.add_constant(X2).astype('float64')
    mask2 = (~y.isna()) & (~X2.isna().any(axis=1))
    model2 = sm.OLS(y.loc[mask2].astype('float64'), X2.loc[mask2]).fit()

    return model0, model1, model2

model0, model1, model2 = regression(first_author_level_data)
model3, model4, model5 = regression(last_author_level_data)

main_vars = [
    'const', 'ai_author', 'author_english_country', 'ai_author_english',
    'author_ins_top_100', 'cumulative_pubs', 'cumulative_cits', 'author_age'
]

models = [model0, model1, model2, model3, model4, model5]

results_table = summary_col(
    models,
    model_names=[
        'first-baseline', 'first-basic', 'first-interaction',
        'last-baseline', 'last-basic', 'last-interaction'
    ],
    stars=False,
    float_format='%.4f',
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    },
    regressor_order=main_vars
)

summary_df = results_table.tables[0].copy()

for j, model in enumerate(models):
    col = summary_df.columns[j]
    for var in main_vars:
        if var in model.params and var in summary_df.index:
            p = model.pvalues[var]
            stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            summary_df.loc[var, col] = f"{model.params[var]:.4f}{stars}"

rows_to_keep = []

for i, row_name in enumerate(summary_df.index):
    if row_name in main_vars:
        rows_to_keep.append(i)
        if i + 1 < len(summary_df):
            rows_to_keep.append(i + 1)
    elif row_name in ['N', 'R2']:
        rows_to_keep.append(i)

summary_df = summary_df.iloc[rows_to_keep]

summary_df.to_csv(
    '../../results/tables/ols_regression_ai_author_results.csv',
    encoding='utf-8-sig'
)

## Productivity change analysis

In [8]:
def regression(data):
    df_work = data.copy()
    y = pd.to_numeric(df_work['pub_change'], errors='coerce')

    control_vars = [
        'author_ins_top_100',
        'cumulative_pubs',
        'cumulative_cits',
        'author_age',
        'author_english_country'
    ]

    X_controls = df_work[control_vars].copy()

    for col in X_controls.columns:
        X_controls[col] = pd.to_numeric(X_controls[col], errors='coerce')

    X_controls['cumulative_pubs'] = np.log1p(X_controls['cumulative_pubs'])
    X_controls['cumulative_cits'] = np.log1p(X_controls['cumulative_cits'])

    subfield_dummies = pd.get_dummies(
        df_work['author_field'],
        prefix='subfield',
        drop_first=True
    ).astype(float)

    X1 = X_controls.copy()
    X1['ai_use_change'] = pd.to_numeric(
        df_work['ai_use_change'],
        errors='coerce'
    )
    X1 = pd.concat([X1, subfield_dummies], axis=1)
    X1 = sm.add_constant(X1).astype('float64')

    mask1 = (~y.isna()) & (~X1.isna().any(axis=1))
    model1 = sm.OLS(
        y.loc[mask1].astype('float64'),
        X1.loc[mask1]
    ).fit()

    X2 = X_controls.copy()
    X2['ai_use_change'] = pd.to_numeric(
        df_work['ai_use_change'],
        errors='coerce'
    )
    X2['ai_use_change_english'] = (
        X2['ai_use_change'] * X2['author_english_country']
    )

    X2 = pd.concat([X2, subfield_dummies], axis=1)
    X2 = sm.add_constant(X2).astype('float64')

    mask2 = (~y.isna()) & (~X2.isna().any(axis=1))
    model2 = sm.OLS(
        y.loc[mask2].astype('float64'),
        X2.loc[mask2]
    ).fit()

    return model1, model2


model1, model2 = regression(first_author_level_data)
model4, model5 = regression(last_author_level_data)

main_vars = [
    'const',
    'ai_use_change',
    'author_english_country',
    'ai_use_change_english',
    'author_ins_top_100',
    'cumulative_pubs',
    'cumulative_cits',
    'author_age'
]

models = [model1, model2, model4, model5]

results_table = summary_col(
    models,
    model_names=[
        'first-basic',
        'first-interaction',
        'last-basic',
        'last-interaction'
    ],
    stars=False,
    float_format='%.4f',
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    },
    regressor_order=main_vars
)

summary_df = results_table.tables[0].copy()

for j, model in enumerate(models):
    col = summary_df.columns[j]

    for var in main_vars:
        if var in model.params and var in summary_df.index:
            p = model.pvalues[var]

            stars = (
                '***' if p < 0.001
                else '**' if p < 0.01
                else '*' if p < 0.05
                else ''
            )

            summary_df.loc[var, col] = (
                f"{model.params[var]:.4f}{stars}"
            )

rows_to_keep = []

for i, row_name in enumerate(summary_df.index):
    if row_name in main_vars:
        rows_to_keep.append(i)

        if i + 1 < len(summary_df):
            rows_to_keep.append(i + 1)

    elif row_name in ['N', 'R2']:
        rows_to_keep.append(i)

summary_df = summary_df.iloc[rows_to_keep]

summary_df.to_csv(
    '../../results/tables/ols_regression_productivity_results.csv',
    encoding='utf-8-sig'
)